In [ ]:
%pip install natsort
import pandas as pd
from pathlib import Path
from scipy.optimize import least_squares
from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import re
from natsort import natsorted

# =============================================================================
# 1. 参数区
# =============================================================================
folder_path = Path(r"D:\毕设数据\20_export_pulse\20_export_pulse\METABatt_Sony_Murata_18650VTC6_007")
SOH_FILENAME_PATTERN = re.compile(r"BM\d+_(\d+(?:\.\d+)?)SOH\.parquet$", flags=re.IGNORECASE)


def extract_soh_from_filename(filename):
    match = SOH_FILENAME_PATTERN.search(filename.strip())
    if match is None:
        raise ValueError(f"无法从文件名解析 SOH: {filename}")
    return float(match.group(1))

# 实验设置的SOC顺序
SOC_ORDER = ["90%", "50%", "10%"]

# cycle内脉冲按照ID划分
REMOVE_PULSE_BEFORE_MIN = 60

# 实测 active 会比 3h 稍大，但不会超过 CYCLE_ACTIVE_LIMIT_HOUR.
# 因此用 CYCLE_ACTIVE_LIMIT_HOUR 作为 time_diff 判断是否进入下一 cycle 的边界。
CYCLE_ACTIVE_LIMIT_HOUR = 4.0

# 设置电流标准差
STD_LIMIT_1P5A = 0.1
STD_LIMIT_3A = 0.1

# R0计算设置
R0_TARGET_AFTER_PAUSE_SEC = 0.5   # pause段最后一个测量点之后外推的R0时间
R0_FIT_POINT_START = 2            # 默认使用有效pulse第2-6点
R0_FIT_POINT_END = 6              # 拟合终点固定为有效pulse第6点
FIRST_TWO_VOLTAGE_EQUAL_ATOL = 1e-9  # 前两点电压视为相同的绝对容差(V)
ZERO_CURRENT_LIMIT = 1e-6
VOLTAGE_JUMP_LIMIT = 1e-3
MAX_PAUSE_TO_PULSE_GAP_SEC = 5.0  # pause末点到pulse首点严格大于5s则剔除
R0_EARLY_WINDOW_FLAG = "后段电流不稳定，R0仅使用早期稳定窗口"

# R0置信度规则：
# “干净标签”的有效R0可按权重1.0使用。
TRUSTED_R0_QUALITY_FLAGS = {
    "正常",
    R0_EARLY_WINDOW_FLAG,
    "首点0且电压跳变"
}

R0_CONFIDENCE_FULL = "完全可信（权重1.0）"
R0_CONFIDENCE_REVIEW = "需复核（权重0.5）"
R0_CONFIDENCE_INVALID = "不可用（权重0.0）"

TIME_DIFF_OUTPUT_COLUMNS = [
    "SOH",
    "SOC",
    "File",
    "Time",
    "Current",
    "Voltage",
    "Zustand",
    "ID",
    "Zustand/Current"
]

# R0及质量检验列
PULSE_OUTPUT_COLUMNS = TIME_DIFF_OUTPUT_COLUMNS + [
    "R0",
    "R0_Target_Time",
    "Pause_to_Pulse_Time_Diff_s",
    "R0_Quality",
    "R0_Confidence"
]


In [ ]:
# =============================================================================
# 2. 读取 parquet
# =============================================================================

parquet_files = natsorted(list(folder_path.rglob("*.parquet")))

if len(parquet_files) == 0:
    raise FileNotFoundError(f"没有在文件夹中找到 parquet 文件: {folder_path}")

df_list = []

for file in parquet_files:
    temp = pd.read_parquet(file)
    temp["File"] = file.name
    temp["SOH"] = extract_soh_from_filename(file.name)

    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

In [ ]:
# =============================================================================
# 3. time diff 主函数
# =============================================================================

def build_time_diff_sequence(df):
    df_td = df.copy()

    # -----------------------------
    # 1. 基础时间、电流、电压处理
    # -----------------------------
    df_td["Time"] = pd.to_datetime(df_td["Time"], utc=True, errors="coerce")
    df_td["Current"] = pd.to_numeric(df_td["Current"], errors="coerce")
    df_td["Voltage"] = pd.to_numeric(df_td["Voltage"], errors="coerce")

    df_td = df_td.dropna(subset=["Time", "Current"]).copy()
    df_td = df_td.sort_values(["File", "Time"]).reset_index(drop=True)

    # -----------------------------
    # 2. 按 time_diff 划分 cycle / SOC
    # -----------------------------
    # 直接比较同一个 File 内相邻时间点的时间差：
    #   time_diff <= CYCLE_ACTIVE_LIMIT_HOUR：仍属于当前 cycle
    #   time_diff >  CYCLE_ACTIVE_LIMIT_HOUR：说明中间经过 pause，进入下一个 cycle
    df_td["time_diff_hour"] = (
        df_td.groupby("File")["Time"].diff() / pd.Timedelta(hours=1)
    )

    df_td["is_new_cycle"] = (
        df_td["time_diff_hour"].isna()
        | (df_td["time_diff_hour"] > CYCLE_ACTIVE_LIMIT_HOUR)
    )

    df_td["cycle_id"] = (
        df_td.groupby("File")["is_new_cycle"]
        .cumsum()
        .astype(int)
    )

    df_td["SOC"] = df_td["cycle_id"].map(
        lambda cycle_id: SOC_ORDER[(cycle_id - 1) % len(SOC_ORDER)]
    )

    cycle_start_time = df_td.groupby(["File", "cycle_id"])["Time"].transform("min")

    df_td["time_from_cycle_start_min"] = (
        df_td["Time"] - cycle_start_time
    ) / pd.Timedelta(minutes=1)

    # -----------------------------
    # 3. 统一 Zustand
    # -----------------------------
    df_td["Zustand"] = df_td["Zustand"].astype(str)

    df_td.loc[
        df_td["Zustand"].str.startswith("DCH", na=False),
        "Zustand"
    ] = "DCH"

    df_td.loc[
        df_td["Zustand"].str.startswith("CHA", na=False),
        "Zustand"
    ] = "CHA"

    # -----------------------------
    # 4. 生成 pulse_segment_id
    # -----------------------------
    df_td["pulse_segment_id"] = (
        df_td["File"].ne(df_td["File"].shift())
        | df_td["Zustand"].ne(df_td["Zustand"].shift())
    ).cumsum()

    # -----------------------------
    # 5. 生成 Zustand/Current，并保留完整 time_diff_sequence
    # -----------------------------
    df_td["Zustand/Current"] = (
        df_td["Zustand"]
        + "/"
        + df_td["Current"].astype(float).round(1).astype(str)
    )

    # 这个表保留所有状态点，后面用于筛选 PAUO
    time_diff_sequence = df_td[TIME_DIFF_OUTPUT_COLUMNS].copy()

    # 只保留 CHA / DCH 作为 pulse_sequence
    pulse_mask = df_td["Zustand"].str.startswith(("CHA", "DCH"), na=False)
    pulse_sequence = df_td[pulse_mask].copy()
    pulse_sequence = pulse_sequence.sort_values(
        ["File", "pulse_segment_id", "Time"]
    )

    # -----------------------------
    # 6. 剔除电流不稳定的 pulse_segment_id
    # -----------------------------
    def get_effective_start_pos(group):
    # 返回pulse段内第一个非零有效电流点的位置。
        current_abs = group["Current"].abs().to_numpy()
        non_zero_pos = np.flatnonzero(current_abs > ZERO_CURRENT_LIMIT)

        if len(non_zero_pos) == 0:
            return None

        return int(non_zero_pos[0])

    def is_bad_current_segment(group):
        group = group.sort_values("Time")

        effective_start_pos = get_effective_start_pos(group)
        if effective_start_pos is None:
            return True

        # 如果首个测量点 Current == 0，则从第一个非零有效点开始判断稳定性
        current_values = group["Current"].iloc[effective_start_pos:]

        current_abs_level = round(current_values.abs().iloc[0], 1)
        current_std = current_values.std()

        if current_abs_level == 1.5:
            return current_std > STD_LIMIT_1P5A

        if current_abs_level == 3.0:
            return current_std > STD_LIMIT_3A

        return True

    # 不再因为pulse后段电流下降而提前删除整段数据。
    # R0是否可计算，改为在下面仅依据实际拟合窗口内的电流稳定性判断。

    # -----------------------------
    # 7. 计算R0，并且每个 pulse_segment_id 只取一个代表点
    # -----------------------------
    def add_quality(base_quality, new_quality):
        if base_quality == "正常":
            return new_quality
        return base_quality + "；" + new_quality

    def calculate_r0_for_segment(group, effective_start_pos):
        group = group.sort_values("Time")

        r0_result = {
            "R0": np.nan,
            "R0_Target_Time": pd.NaT,
            "Pause_to_Pulse_Time_Diff_s": np.nan,
            "R0_Quality": "正常"
        }

        file_name = group["File"].iloc[0]
        pulse_start_time = group["Time"].iloc[0]
        first_current = group["Current"].iloc[0]

        previous_pause = df_td[
            (df_td["File"] == file_name)
            & (df_td["Time"] < pulse_start_time)
            & (df_td["Zustand"].str.startswith("PAU", na=False))
        ].sort_values("Time").tail(1)

        if previous_pause.empty:
            r0_result["R0_Quality"] = "无法计算R0：无前置pause点"
            return r0_result

        pause_time = previous_pause["Time"].iloc[0]
        pause_voltage = previous_pause["Voltage"].iloc[0]

        pause_to_pulse_gap_s = (pulse_start_time - pause_time).total_seconds()
        r0_result["Pause_to_Pulse_Time_Diff_s"] = pause_to_pulse_gap_s

        target_time = pause_time + pd.Timedelta(seconds=R0_TARGET_AFTER_PAUSE_SEC)
        r0_result["R0_Target_Time"] = target_time

        # 大于5s：标记为无效，不再进行长距离反向外推
        if pause_to_pulse_gap_s > MAX_PAUSE_TO_PULSE_GAP_SEC:
            r0_result["R0_Quality"] = "pause结束点到pulse首点时间差>5s"
            return r0_result

        # pulse段第一个点为0时：无论voltage是否跳变，R0计算都从第一个非零有效点开始；
        # 若voltage已经跳变，则额外给质量标记。
        if abs(first_current) <= ZERO_CURRENT_LIMIT:
            first_voltage = group["Voltage"].iloc[0]

            if (
                pd.notna(first_voltage)
                and pd.notna(pause_voltage)
                and abs(first_voltage - pause_voltage) > VOLTAGE_JUMP_LIMIT
            ):
                r0_result["R0_Quality"] = "首点0且电压跳变"

        # 默认使用有效pulse第2-6点进行线性拟合。
        # 若有效pulse第1点与第2点的电压相同（可能是重复采样），
        # 则跳过前两点，改用第3-6点进行线性外推。
        fit_point_start = R0_FIT_POINT_START

        if effective_start_pos + 1 < len(group):
            first_pulse_voltage = group["Voltage"].iloc[effective_start_pos]
            second_pulse_voltage = group["Voltage"].iloc[effective_start_pos + 1]

            first_two_voltage_equal = (
                pd.notna(first_pulse_voltage)
                and pd.notna(second_pulse_voltage)
                and np.isclose(
                    float(first_pulse_voltage),
                    float(second_pulse_voltage),
                    rtol=0.0,
                    atol=FIRST_TWO_VOLTAGE_EQUAL_ATOL
                )
            )

            if first_two_voltage_equal:
                fit_point_start = 3

        fit_start_pos = effective_start_pos + fit_point_start - 1
        fit_end_pos = effective_start_pos + R0_FIT_POINT_END

        fit_points = group.iloc[fit_start_pos:fit_end_pos].dropna(
            subset=["Time", "Voltage", "Current"]
        )

        if len(fit_points) < 2:
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：拟合点不足"
            )
            return r0_result

        # R0只要求实际参与2-6点（或3-6点）拟合的早期窗口电流稳定。
        fit_current_values = fit_points["Current"]
        fit_current_abs_level = round(fit_current_values.abs().iloc[0], 1)
        fit_current_std = fit_current_values.std()

        if fit_current_abs_level == 1.5:
            fit_current_unstable = fit_current_std > STD_LIMIT_1P5A
        elif fit_current_abs_level == 3.0:
            fit_current_unstable = fit_current_std > STD_LIMIT_3A
        else:
            fit_current_unstable = True

        if fit_current_unstable:
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：早期拟合窗口电流不稳定"
            )
            return r0_result

        # 若整段电流不稳定、但早期拟合窗口稳定，仍计算R0并添加专用flag。
        if is_bad_current_segment(group):
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                R0_EARLY_WINDOW_FLAG
            )

        effective_current = group["Current"].iloc[effective_start_pos]

        if abs(effective_current) <= ZERO_CURRENT_LIMIT:
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：无非零有效电流"
            )
            return r0_result

        if pd.isna(pause_voltage):
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：pause电压缺失"
            )
            return r0_result
        
        # 判断采样点距离pulse起点的距离
        x_sec = (fit_points["Time"] - target_time) / pd.Timedelta(seconds=1)

        y_voltage = fit_points["Voltage"].astype(float)

        slope, intercept = np.polyfit(x_sec.to_numpy(), y_voltage.to_numpy(), 1)
        extrapolated_voltage = intercept

        r0_result["R0"] = abs(
            (extrapolated_voltage - pause_voltage) / effective_current
        )

        return r0_result

    selected_indices = []
    r0_results = {}

    for _, group in pulse_sequence.groupby(["File", "pulse_segment_id"], sort=False):
        group = group.sort_values("Time")

        effective_start_pos = get_effective_start_pos(group)
        if effective_start_pos is None:
            continue

        # 保留原逻辑：每段用“有效pulse起点后的第2个测量点”记录；
        # 如果点数不够，则退回到有效pulse起点本身。
        record_pos = effective_start_pos + 1
        if record_pos >= len(group):
            record_pos = effective_start_pos

        record_index = group.index[record_pos]
        selected_indices.append(record_index)

        r0_results[record_index] = calculate_r0_for_segment(
            group,
            effective_start_pos
        )

    pulse_sequence = pulse_sequence.loc[selected_indices].copy()

    r0_result_columns = [
        "R0",
        "R0_Target_Time",
        "Pause_to_Pulse_Time_Diff_s",
        "R0_Quality"
    ]

    for column in r0_result_columns:
        pulse_sequence[column] = pulse_sequence.index.map(
            lambda idx, col=column: r0_results[idx][col]
        )

    pulse_sequence = pulse_sequence.reset_index(drop=True)

    # -----------------------------
    # 8. 生成逐条R0置信度
    # -----------------------------
    def classify_r0_confidence(row):
        r0_value = row["R0"]
        quality_text = str(row["R0_Quality"]).strip()

        # 没有有效R0时，无论带有什么flag，都不能参与R0分析。
        if pd.isna(r0_value) or not np.isfinite(r0_value):
            return R0_CONFIDENCE_INVALID

        flags = {
            flag.strip()
            for flag in quality_text.split("；")
            if flag.strip()
        }

        # 只要一条记录所含flag全部属于已验证的干净标签，就给权重1.0。
        if flags and flags.issubset(TRUSTED_R0_QUALITY_FLAGS):
            return R0_CONFIDENCE_FULL

        # 将来若新增了仍可计算R0、但尚未验证的flag，先标记为需复核。
        return R0_CONFIDENCE_REVIEW

    pulse_sequence["R0_Confidence"] = pulse_sequence.apply(
        classify_r0_confidence,
        axis=1
    )

    # -----------------------------
    # 9. 只保留最终输出列
    # -----------------------------
    pulse_sequence = pulse_sequence[PULSE_OUTPUT_COLUMNS].copy()

    return pulse_sequence, time_diff_sequence

pulse_sequence, time_diff_sequence = build_time_diff_sequence(df)

# 中间结果预览；统一的R0_Quality统计放在最终绘图cell中，避免重复输出。
display(pulse_sequence)


KeyboardInterrupt: 

In [ ]:
# R0 计算部分
# -----------------------------
# 9. 筛选脉冲/清除1.5A下的DCH脉冲
# -----------------------------
def filter_pulse(df):

    pulse_sequence_filter = df[~df["Zustand/Current"].isin(["DCH/-1.5"])].copy()
    return pulse_sequence_filter

filtered_pulse = filter_pulse(pulse_sequence)
display(filtered_pulse)


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current,R0,R0_Target_Time,Pause_to_Pulse_Time_Diff_s,R0_Quality,R0_Confidence
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.740000+00:00,1.499155,4.127363,CHA,12_21,CHA/1.5,0.025070,2024-11-12 13:04:35.250000+00:00,0.76,正常,完全可信（权重1.0）
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.910000+00:00,-2.998064,4.009585,DCH,12_25,DCH/-3.0,0.024941,2024-11-12 14:05:18.390000+00:00,0.81,正常,完全可信（权重1.0）
3,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 15:06:22.310000+00:00,2.995714,4.165722,CHA,12_29,CHA/3.0,0.024550,2024-11-12 15:06:21.920000+00:00,0.79,后段电流不稳定，R0仅使用早期稳定窗口,完全可信（权重1.0）
5,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.940000+00:00,1.499604,3.805598,CHA,12_39,CHA/1.5,0.018330,2024-11-12 20:56:39.410000+00:00,0.81,正常,完全可信（权重1.0）
6,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:23.250000+00:00,-2.999144,3.719279,DCH,12_43,DCH/-3.0,0.018422,2024-11-12 21:57:22.670000+00:00,0.85,正常,完全可信（权重1.0）
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:49.530000+00:00,-2.999504,3.717166,DCH,9_44,DCH/-3.0,0.017919,2024-10-23 22:29:48.980000+00:00,0.84,正常,完全可信（权重1.0）
279,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.820000+00:00,2.999671,3.831503,CHA,9_48,CHA/3.0,0.017858,2024-10-23 23:30:52.300000+00:00,0.81,正常,完全可信（权重1.0）
281,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:32.010000+00:00,1.498525,3.332511,CHA,9_58,CHA/1.5,0.019617,2024-10-24 05:23:31.420000+00:00,0.88,正常,完全可信（权重1.0）
282,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:15.220000+00:00,-2.996265,3.241513,DCH,9_62,DCH/-3.0,0.019862,2024-10-24 06:24:14.600000+00:00,0.86,正常,完全可信（权重1.0）


In [ ]:
# =============================================================================
# 10. R0结果按flag输出 + 按SOC / pulse / current分类绘图
# =============================================================================
# 说明：
#   1. R0 本身已经在 build_time_diff_sequence(df) 中计算完成；
#   2. 这里基于 filtered_pulse 输出 R0 结果和 flag 统计；
#   3. R0 单位从 Ohm 转为 mOhm；
#   4. 绘图分类方式参考：
#      10% / 50% / 90% SOC 分成三个子图；
#      每条线按 SOC + CHA/DCH + 电流大小 分类。

# -----------------------------
# 1. 选择R0结果来源
# -----------------------------
if "filtered_pulse" in globals():
    r0_source = filtered_pulse.copy()
elif "pulse_sequence" in globals():
    r0_source = pulse_sequence.copy()
else:
    raise NameError("请先运行前面的cell，生成 pulse_sequence 或 filtered_pulse。")

required_columns = [
    "SOH",
    "SOC",
    "Time",
    "Current",
    "Voltage",
    "Zustand",
    "Zustand/Current",
    "R0",
    "R0_Target_Time",
    "Pause_to_Pulse_Time_Diff_s",
    "R0_Quality"
]

missing_columns = [col for col in required_columns if col not in r0_source.columns]
if missing_columns:
    raise KeyError(f"R0结果缺少必要列: {missing_columns}")

# -----------------------------
# 2. 整理R0结果
# -----------------------------
r0_result = r0_source.copy()

r0_result["SOH"] = pd.to_numeric(r0_result["SOH"], errors="coerce")
r0_result["Current"] = pd.to_numeric(r0_result["Current"], errors="coerce")
r0_result["R0"] = pd.to_numeric(r0_result["R0"], errors="coerce")

# R0原单位为 Ohm，这里转换为 mOhm，方便和图里的量级一致
r0_result["R0_mOhm"] = r0_result["R0"] * 1000

r0_result["R0_Quality"] = (
    r0_result["R0_Quality"]
    .fillna("缺失")
    .astype(str)
    .str.strip()
)

r0_result["Current_abs_A"] = r0_result["Current"].abs().round(1)

r0_result["Current_Label"] = r0_result["Current_abs_A"].map(
    lambda x: f"{x:.1f}A" if pd.notna(x) else "UnknownA"
)

r0_result["SOC_pulse_current"] = (
    r0_result["SOC"].astype(str)
    + " "
    + r0_result["Zustand"].astype(str)
    + " "
    + r0_result["Current_Label"]
)

r0_result["R0_Is_Valid"] = (
    r0_result["R0_mOhm"].notna()
    & np.isfinite(r0_result["R0_mOhm"])
)

# 逐条记录的R0置信度。即使前面cell尚未重跑，这里也会重新生成。
def classify_r0_confidence(row):
    if not row["R0_Is_Valid"]:
        return R0_CONFIDENCE_INVALID

    flags = {
        flag.strip()
        for flag in str(row["R0_Quality"]).split("；")
        if flag.strip()
    }

    if flags and flags.issubset(TRUSTED_R0_QUALITY_FLAGS):
        return R0_CONFIDENCE_FULL

    return R0_CONFIDENCE_REVIEW

r0_result["R0_Confidence"] = r0_result.apply(
    classify_r0_confidence,
    axis=1
)

result_display_columns = [
    "SOH",
    "SOC",
    "Time",
    "Zustand",
    "Current",
    "Voltage",
    "R0",
    "R0_mOhm",
    "R0_Target_Time",
    "Pause_to_Pulse_Time_Diff_s",
    "R0_Quality",
    "R0_Confidence",
    "SOC_pulse_current",
    "File"
]

print("R0结果：按 R0_Quality / SOC / SOH 排序")
display(
    r0_result[result_display_columns]
    .sort_values(
        ["R0_Quality", "SOC", "SOH", "Zustand", "Current"],
        ascending=[True, True, False, True, True]
    )
    .reset_index(drop=True)
)

# -----------------------------
# 3. R0_Quality统计
# -----------------------------
# 若一条记录包含多个以“；”分隔的flag，会拆开后分别统计。
r0_quality_summary = (
    r0_result
    .assign(R0_Quality_Flag=r0_result["R0_Quality"].str.split("；"))
    .explode("R0_Quality_Flag")
)

r0_quality_summary["R0_Quality_Flag"] = (
    r0_quality_summary["R0_Quality_Flag"]
    .fillna("缺失")
    .astype(str)
    .str.strip()
)

r0_quality_summary = (
    r0_quality_summary
    .groupby("R0_Quality_Flag", dropna=False)
    .agg(
        Flag_Count=("R0_Quality_Flag", "size"),
        Valid_R0_Count=("R0_Is_Valid", "sum")
    )
    .reset_index()
    .sort_values("Flag_Count", ascending=False)
)

r0_quality_summary["Percent_of_segments"] = (
    r0_quality_summary["Flag_Count"] / len(r0_result) * 100
)

# flag级别的建议置信度：
# 3（正常）、1（后段不稳但早期窗口稳定）、4（首点0且电压跳变）
# 都作为干净标签，建议权重1.0。
r0_quality_summary["R0_Confidence"] = np.select(
    [
        (
            r0_quality_summary["R0_Quality_Flag"].isin(
                TRUSTED_R0_QUALITY_FLAGS
            )
            & r0_quality_summary["Valid_R0_Count"].gt(0)
        ),
        r0_quality_summary["Valid_R0_Count"].eq(0)
    ],
    [
        R0_CONFIDENCE_FULL,
        R0_CONFIDENCE_INVALID
    ],
    default=R0_CONFIDENCE_REVIEW
)

print("R0_Quality统计：")
display(r0_quality_summary)

# -----------------------------
# 4. 按图示方式绘图：SOH vs R0，分SOC子图，按 SOC / pulse / current 分线
# -----------------------------
# 绘制正常R0，以及“后段电流不稳定但早期拟合窗口稳定”的有效R0
r0_plot_data = r0_result[
    r0_result["R0_Is_Valid"]
    & r0_result["R0_Quality"].isin([
        "正常",
        R0_EARLY_WINDOW_FLAG
    ])
].copy()

if r0_plot_data.empty:
    print("没有可绘制的有效R0数据。")
else:
    soc_plot_order = ["10%", "50%", "90%"]
    available_soc_order = [
        soc for soc in soc_plot_order
        if soc in r0_plot_data["SOC"].astype(str).unique()
    ]

    # 如果出现了不在默认顺序中的SOC，也保留在后面
    extra_soc = [
        soc for soc in r0_plot_data["SOC"].astype(str).unique()
        if soc not in available_soc_order
    ]

    available_soc_order = available_soc_order + sorted(extra_soc)

    fig = make_subplots(
        rows=1,
        cols=len(available_soc_order),
        subplot_titles=[f"{soc} SOC" for soc in available_soc_order],
        shared_yaxes=False,
        horizontal_spacing=0.08
    )

    shown_legend = set()

    for col_idx, soc in enumerate(available_soc_order, start=1):
        soc_data = r0_plot_data[
            r0_plot_data["SOC"].astype(str) == soc
        ].copy()

        # 按SOH从高到低排序，配合反向x轴，视觉上和示例图一致
        soc_data = soc_data.sort_values(
            ["SOC_pulse_current", "SOH"],
            ascending=[True, False]
        )

        for label, group in soc_data.groupby("SOC_pulse_current", sort=True):
            group = group.sort_values("SOH", ascending=False)

            fig.add_trace(
                go.Scatter(
                    x=group["SOH"],
                    y=group["R0_mOhm"],
                    mode="lines+markers",
                    name=label,
                    legendgroup=label,
                    showlegend=label not in shown_legend
                ),
                row=1,
                col=col_idx
            )

            shown_legend.add(label)

        fig.update_xaxes(
            title_text="SOH (%)",
            autorange="reversed",
            row=1,
            col=col_idx
        )

        fig.update_yaxes(
            title_text="R0 (mOhm)",
            row=1,
            col=col_idx
        )

    fig.update_layout(
        title="SOH vs SOC 10%, 50%, 90% at R0",
        legend_title_text="SOC / pulse / current",
        width=1500,
        height=600,
        template="plotly_white"
    )

    fig.show()


R0结果：按 R0_Quality / SOC / SOH 排序


,SOH,SOC,Time,Zustand,Current,Voltage,R0,R0_mOhm,R0_Target_Time,Pause_to_Pulse_Time_Diff_s,R0_Quality,R0_Confidence,SOC_pulse_current,File
0,81.7,50%,2025-06-26 20:02:05.060000+00:00,CHA,2.999851,3.879424,NaN,NaN,2025-06-26 20:01:53.770000+00:00,11.66,pause结束点到pulse首点时间差>5s,不可用（权重0.0）,50% CHA 3.0A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM34_...
1,81.7,90%,2025-06-26 12:20:22.870000+00:00,CHA,2.370311,4.199966,NaN,NaN,2025-06-26 12:20:04.430000+00:00,18.80,pause结束点到pulse首点时间差>5s,不可用（权重0.0）,90% CHA 2.4A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM34_...
2,90.3,90%,2024-11-12 15:06:22.310000+00:00,CHA,2.995714,4.165722,0.024550,24.550339,2024-11-12 15:06:21.920000+00:00,0.79,后段电流不稳定，R0仅使用早期稳定窗口,完全可信（权重1.0）,90% CHA 3.0A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...
3,89.1,90%,2024-12-01 15:26:19.060000+00:00,CHA,2.997153,4.174616,0.026980,26.979597,2024-12-01 15:26:18.550000+00:00,0.80,后段电流不稳定，R0仅使用早期稳定窗口,完全可信（权重1.0）,90% CHA 3.0A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM14_...
4,88.0,90%,2024-12-19 01:29:21.090000+00:00,CHA,2.995175,4.176729,0.027769,27.769312,2024-12-19 01:29:20.630000+00:00,0.80,后段电流不稳定，R0仅使用早期稳定窗口,完全可信（权重1.0）,90% CHA 3.0A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM16_...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,78.6,90%,2025-11-09 20:12:28.080000+00:00,CHA,1.493757,4.145041,0.032455,32.455231,2025-11-09 20:12:27.730000+00:00,0.75,首点0且电压跳变,完全可信（权重1.0）,90% CHA 1.5A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM48_...
212,78.6,90%,2025-11-09 21:13:10.940000+00:00,DCH,-2.991047,3.982345,0.032661,32.660834,2025-11-09 21:13:10.630000+00:00,0.74,首点0且电压跳变,完全可信（权重1.0）,90% DCH 3.0A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM48_...
213,77.8,90%,2025-12-14 13:48:16.550000+00:00,CHA,1.496726,4.151712,0.039542,39.542019,2025-12-14 13:48:16.140000+00:00,0.76,首点0且电压跳变,完全可信（权重1.0）,90% CHA 1.5A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM52_...
214,79.4,90%,2025-10-04 17:07:30.340000+00:00,CHA,2.597305,4.200300,NaN,NaN,2025-10-04 17:07:29.990000+00:00,0.75,首点0且电压跳变；无法计算R0：早期拟合窗口电流不稳定,不可用（权重0.0）,90% CHA 2.6A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM44_...


R0_Quality统计：


,R0_Quality_Flag,Flag_Count,Valid_R0_Count,Percent_of_segments,R0_Confidence
3,正常,179,179,82.870370,完全可信（权重1.0）
4,首点0且电压跳变,16,14,7.407407,完全可信（权重1.0）
2,无法计算R0：早期拟合窗口电流不稳定,11,0,5.092593,不可用（权重0.0）
1,后段电流不稳定，R0仅使用早期稳定窗口,10,10,4.629630,完全可信（权重1.0）
0,pause结束点到pulse首点时间差>5s,2,0,0.925926,不可用（权重0.0）


In [ ]:
# =============================================================================
# 12. 仅拟合 10% SOC：二阶 RC + finite-length Warburg Open
# =============================================================================
# 电路拓扑：
#   R0 + (R1 || C1) + (R2 || C2) + W_open
#
# Warburg Open（反射/阻塞边界）：
#   Z_W(s) = Rd * coth(sqrt(s*td)) / sqrt(s*td)
#
# 使用恒等式：
#   coth(z)/z = 1/z^2 + 2 * sum_{n=1..inf} 1/(z^2 + n^2*pi^2)
#
# 因而可写成一个积分状态 + N 个一阶指数状态：
#   dVw0/dt = (Rd/td) * I
#   dVwn/dt = -(n^2*pi^2/td)*Vwn + (2*Rd/td)*I
#
# 截断到 N=3~5 后，每个扩散模态仍是标准指数响应，可直接使用
# scipy.optimize.least_squares 拟合，不需要分数阶数值算法。
#
# 重要单位：
#   r0_result["R0"] 的原始单位是 Ohm；
#   r0_result["R0_mOhm"] 的单位是 mOhm。
# =============================================================================

TARGET_SOC = "10%"

WARBURG_MAIN_TERMS = 5
WARBURG_CHECK_TERMS = 3
WARBURG_NOISE_FLOOR_MV = 0.5   # 可按设备实际电压噪声修改

# 旧的 td >= 1.5*tau2 软惩罚默认关闭。
# 原因：finite-length Warburg 的指数模态时间常数为
#   tau_W,n = td / (n^2*pi^2)
# 因此直接比较 td 和 tau2 不能正确表达动态先后关系，并会扭曲优化结果。
USE_DIFFUSION_SEPARATION_PENALTY = False

# Stage 2 联合拟合中，pulse 与后续 PAUO 的总权重。
# 两段都完整纳入；权重按各段的时间支撑归一化。
PULSE_WEIGHT_TOTAL = 0.55
PAUO_WEIGHT_TOTAL = 0.45

# 仅针对 10% SOC 的二阶 RC + Warburg 初始值。
# R1_fraction / R2_fraction 表示总动态电阻初值中分配给两个 RC 支路的比例，
# 其余部分分配给 Warburg 电阻 Rd。
RC2_WARBURG_INITIAL = {
    "R1_fraction": 0.15,
    "R2_fraction": 0.45,
    "tau1": 12.0,
    "tau2": 400.0,
    "td": 1800.0,
}

# 仅针对 10% SOC 的参数边界。
# R2_max 敏感性测试：原值 0.80，临时放宽到 5.00，用于判断 R2:upper
# 究竟是边界设置过紧，还是模型本身存在无上界的可辨识性问题。
RC2_WARBURG_BOUNDS = {
    "R1_max": 0.10,
    "R2_max": 5.00,
    "Rd_max": 0.80,
    "tau1_min": 0.60,
    "tau1_max": 300.0,  
    "tau2_min": 5.0,
    "tau2_max": np.inf,
    "td_min": 5.0,
    "td_max": 30000.0,
}


def _is_pulse_state(value):
    return str(value).strip().startswith(("CHA", "DCH"))


def _r0_ohm_from_row(row):
    """统一把 R0 转为 Ohm，优先使用显式的 R0_mOhm 列。"""
    r0_mohm = pd.to_numeric(row.get("R0_mOhm", np.nan), errors="coerce")
    if pd.notna(r0_mohm) and np.isfinite(r0_mohm):
        return float(r0_mohm) / 1000.0

    r0_ohm = pd.to_numeric(row.get("R0", np.nan), errors="coerce")
    if pd.notna(r0_ohm) and np.isfinite(r0_ohm):
        return float(r0_ohm)

    return np.nan


def _time_support_weights(t_s, mask, total_weight):
    """
    按每个采样点代表的时间跨度分配权重，并把该片段总平方权重
    归一化为 total_weight。这样 pulse 与长 PAUO 尾段都能稳定参与拟合。
    """
    t_s = np.asarray(t_s, dtype=float)
    mask = np.asarray(mask, dtype=bool)
    support = np.zeros_like(t_s, dtype=float)

    if len(t_s) == 0 or not mask.any():
        return support

    if len(t_s) == 1:
        support[0] = 1.0
    else:
        dt = np.diff(t_s)
        positive_dt = dt[dt > 0]
        fallback = float(np.median(positive_dt)) if len(positive_dt) else 1.0

        support[0] = dt[0] if dt[0] > 0 else fallback
        support[-1] = dt[-1] if dt[-1] > 0 else fallback
        if len(t_s) > 2:
            support[1:-1] = 0.5 * (
                np.maximum(dt[:-1], 0.0) + np.maximum(dt[1:], 0.0)
            )

        support[support <= 0] = fallback

        positive_support = support[support > 0]
        cap = np.quantile(positive_support, 0.95) if len(positive_support) else fallback
        support = np.minimum(support, max(float(cap), fallback))

    segment_support = support[mask]
    denom = float(segment_support.sum())

    weights = np.zeros_like(t_s, dtype=float)
    if denom > 0:
        weights[mask] = np.sqrt(total_weight * segment_support / denom)

    return weights


def build_stage2_fit_windows(time_diff_sequence, r0_result):
    """
    构造 Stage 2 拟合窗口：
      - 前一个 PAUO 末端点只用于确定初始 OCV；
      - 拟合数据从 pulse 首个有效点开始；
      - 当前 pulse 全段 + 紧随其后的完整 PAUO 全段联合拟合。
    """
    seq = time_diff_sequence.copy()
    seq["Zustand_clean"] = (
        seq["Zustand"]
        .astype(str)
        .str.strip()
        .str.replace(r"\*+$", "", regex=True)
    )
    seq["Time_dt"] = pd.to_datetime(seq["Time"], utc=True, errors="coerce")
    seq["Current"] = pd.to_numeric(seq["Current"], errors="coerce")
    seq["Voltage"] = pd.to_numeric(seq["Voltage"], errors="coerce")

    seq = (
        seq
        .dropna(subset=["File", "Time_dt", "Current", "Voltage"])
        .sort_values(["File", "Time_dt"])
        .reset_index(drop=True)
    )

    seq["segment_id"] = (
        seq["File"].ne(seq["File"].shift())
        | seq["Zustand_clean"].ne(seq["Zustand_clean"].shift())
    ).cumsum()

    seq_id_parts = seq["ID"].astype(str).str.extract(r"^(.*)_(\d+)$")
    seq["id_prefix"] = seq_id_parts[0]
    seq["id_number"] = pd.to_numeric(seq_id_parts[1], errors="coerce")

    # 本 notebook 只处理 10% SOC；其他 SOC 已在单独 notebook 中完成。
    r0_table = r0_result[
        r0_result["SOC"].astype(str).str.strip().eq(TARGET_SOC)
    ].copy()
    r0_table["Time_dt"] = pd.to_datetime(
        r0_table["Time"], utc=True, errors="coerce"
    )

    windows = []
    skipped_rows = []

    for _, row in r0_table.iterrows():
        file_name = row.get("File")
        pulse_time = row.get("Time_dt")
        r0_ohm = _r0_ohm_from_row(row)

        skip_reason = None

        if pd.isna(pulse_time):
            skip_reason = "R0记录时间无效"
        elif not np.isfinite(r0_ohm):
            skip_reason = "R0无效"

        file_seq = seq[seq["File"].eq(file_name)].copy()
        if skip_reason is None and file_seq.empty:
            skip_reason = "找不到对应文件数据"

        if skip_reason is not None:
            skipped = row.drop(labels=["Time_dt"], errors="ignore").to_dict()
            skipped["Stage2_Skip_Reason"] = skip_reason
            skipped_rows.append(skipped)
            continue

        pulse_candidates = file_seq[
            file_seq["Time_dt"].eq(pulse_time)
            & file_seq["Zustand"].map(_is_pulse_state)
        ]

        if pulse_candidates.empty:
            skipped = row.drop(labels=["Time_dt"], errors="ignore").to_dict()
            skipped["Stage2_Skip_Reason"] = "找不到R0记录对应的pulse片段"
            skipped_rows.append(skipped)
            continue

        pulse_segment_id = pulse_candidates.iloc[0]["segment_id"]
        pulse_df = file_seq[
            file_seq["segment_id"].eq(pulse_segment_id)
            & file_seq["Zustand"].map(_is_pulse_state)
        ].copy()
        pulse_df = pulse_df.sort_values("Time_dt").reset_index(drop=True)

        while (
            len(pulse_df) >= 2
            and abs(float(pulse_df["Current"].iloc[0])) <= ZERO_CURRENT_LIMIT
        ):
            pulse_df = pulse_df.iloc[1:].reset_index(drop=True)

        if len(pulse_df) < 5:
            skipped = row.drop(labels=["Time_dt"], errors="ignore").to_dict()
            skipped["Stage2_Skip_Reason"] = "pulse有效点少于5个"
            skipped_rows.append(skipped)
            continue

        pulse_start_time = pulse_df["Time_dt"].iloc[0]
        pulse_end_time = pulse_df["Time_dt"].iloc[-1]

        prev_pauo = file_seq[
            file_seq["Time_dt"].lt(pulse_start_time)
            & file_seq["Zustand_clean"].eq("PAUO")
        ].copy()

        if prev_pauo.empty:
            skipped = row.drop(labels=["Time_dt"], errors="ignore").to_dict()
            skipped["Stage2_Skip_Reason"] = "无前置PAUO点"
            skipped_rows.append(skipped)
            continue

        prev_pauo_point = prev_pauo.sort_values("Time_dt").iloc[-1]

        id_parts = str(row.get("ID", "")).rsplit("_", 1)
        if len(id_parts) != 2:
            skipped = row.drop(labels=["Time_dt"], errors="ignore").to_dict()
            skipped["Stage2_Skip_Reason"] = "ID无法解析"
            skipped_rows.append(skipped)
            continue

        pulse_id_prefix = id_parts[0]
        pulse_id_number = pd.to_numeric(id_parts[1], errors="coerce")
        if pd.isna(pulse_id_number):
            skipped = row.drop(labels=["Time_dt"], errors="ignore").to_dict()
            skipped["Stage2_Skip_Reason"] = "ID序号无法解析"
            skipped_rows.append(skipped)
            continue

        next_pauo_df = file_seq[
            file_seq["Time_dt"].gt(pulse_end_time)
            & file_seq["Zustand_clean"].eq("PAUO")
            & file_seq["id_prefix"].eq(pulse_id_prefix)
            & file_seq["id_number"].eq(pulse_id_number + 1)
        ].copy()
        next_pauo_df = next_pauo_df.sort_values("Time_dt").reset_index(drop=True)

        if len(next_pauo_df) < 5:
            skipped = row.drop(labels=["Time_dt"], errors="ignore").to_dict()
            skipped["Stage2_Skip_Reason"] = "后续PAUO有效点少于5个"
            skipped_rows.append(skipped)
            continue

        fit_df = pd.concat(
            [pulse_df, next_pauo_df],
            ignore_index=True,
            sort=False,
        ).sort_values("Time_dt").reset_index(drop=True)

        t_s = (
            fit_df["Time_dt"] - pulse_start_time
        ).dt.total_seconds().to_numpy(dtype=float)
        current_a = fit_df["Current"].to_numpy(dtype=float)
        voltage_v = fit_df["Voltage"].to_numpy(dtype=float)

        pulse_mask = fit_df["Zustand"].map(_is_pulse_state).to_numpy(dtype=bool)
        pauo_mask = fit_df["Zustand_clean"].eq("PAUO").to_numpy(dtype=bool)

        weights = (
            _time_support_weights(t_s, pulse_mask, PULSE_WEIGHT_TOTAL)
            + _time_support_weights(t_s, pauo_mask, PAUO_WEIGHT_TOTAL)
        )

        pulse_duration_s = max(
            float((pulse_end_time - pulse_start_time).total_seconds()), 0.0
        )
        pauo_duration_s = max(
            float(
                (
                    next_pauo_df["Time_dt"].iloc[-1]
                    - next_pauo_df["Time_dt"].iloc[0]
                ).total_seconds()
            ),
            0.0,
        )

        windows.append({
            "row": row.drop(labels=["Time_dt"], errors="ignore").to_dict(),
            "fit_df": fit_df,
            "t_s": t_s,
            "current_a": current_a,
            "voltage_v": voltage_v,
            "weights": weights,
            "pulse_mask": pulse_mask,
            "pauo_mask": pauo_mask,
            "ocv_before": float(prev_pauo_point["Voltage"]),
            "r0_ohm": r0_ohm,
            "pulse_duration_s": pulse_duration_s,
            "pauo_duration_s": pauo_duration_s,
        })

    skipped_df = pd.DataFrame(skipped_rows)
    return windows, skipped_df


def _simulate_rc_state(t_s, current_a, resistance_ohm, tau_s):
    """一阶 R||C 支路在分段常电流下的精确离散更新。"""
    t_s = np.asarray(t_s, dtype=float)
    current_a = np.asarray(current_a, dtype=float)
    voltage_state = np.zeros(len(t_s), dtype=float)

    tau_s = max(float(tau_s), 1e-12)

    for k in range(1, len(t_s)):
        dt = max(float(t_s[k] - t_s[k - 1]), 0.0)
        current_interval = 0.5 * (current_a[k - 1] + current_a[k])
        decay = np.exp(-dt / tau_s)
        voltage_state[k] = (
            decay * voltage_state[k - 1]
            + resistance_ohm * (1.0 - decay) * current_interval
        )

    return voltage_state


def _simulate_warburg_open_state(
    t_s,
    current_a,
    rd_ohm,
    td_s,
    n_terms,
):
    """
    返回值包含：
      1) n=0 的积分状态：反射边界低频电容行为；
      2) n=1..N 的指数扩散模态。
    """
    t_s = np.asarray(t_s, dtype=float)
    current_a = np.asarray(current_a, dtype=float)

    td_s = max(float(td_s), 1e-12)
    rd_ohm = float(rd_ohm)
    n_terms = int(n_terms)

    voltage_w = np.zeros(len(t_s), dtype=float)
    integral_state = 0.0
    mode_states = np.zeros(n_terms, dtype=float)

    mode_index = np.arange(1, n_terms + 1, dtype=float)
    lambdas = (mode_index ** 2) * (np.pi ** 2) / td_s

    for k in range(1, len(t_s)):
        dt = max(float(t_s[k] - t_s[k - 1]), 0.0)
        current_interval = 0.5 * (current_a[k - 1] + current_a[k])

        integral_state += (rd_ohm / td_s) * current_interval * dt

        decay = np.exp(-lambdas * dt)
        mode_states = (
            decay * mode_states
            + (2.0 * rd_ohm / td_s)
            * ((1.0 - decay) / lambdas)
            * current_interval
        )

        voltage_w[k] = integral_state + mode_states.sum()

    return voltage_w


def _simulate_randles_warburg_open(
    param,
    t_s,
    current_a,
    ocv_before,
    r0_ohm,
    n_terms,
):
    """
    param = [R1_ohm, R2_ohm, Rd_ohm, tau1_s, tau2_s, td_s, OCV_offset_V]
    """
    (
        r1_ohm,
        r2_ohm,
        rd_ohm,
        tau1_s,
        tau2_s,
        td_s,
        ocv_offset_v,
    ) = param

    v_r1 = _simulate_rc_state(t_s, current_a, r1_ohm, tau1_s)
    v_r2 = _simulate_rc_state(t_s, current_a, r2_ohm, tau2_s)
    v_w = _simulate_warburg_open_state(
        t_s, current_a, rd_ohm, td_s, n_terms
    )

    return (
        float(ocv_before)
        + float(ocv_offset_v)
        + current_a * float(r0_ohm)
        + v_r1
        + v_r2
        + v_w
    )


def _warburg_initial_and_bounds(window):
    row = window["row"]

    current_a = window["current_a"]
    voltage_v = window["voltage_v"]
    pulse_mask = window["pulse_mask"]
    ocv_before = window["ocv_before"]
    r0_ohm = window["r0_ohm"]

    pulse_current = np.abs(current_a[pulse_mask])
    i_pulse_abs = float(np.nanmedian(pulse_current)) if len(pulse_current) else np.nan

    pulse_voltage = voltage_v[pulse_mask]
    tail_count = min(5, len(pulse_voltage))
    v_pulse_tail = (
        float(np.nanmedian(pulse_voltage[-tail_count:]))
        if tail_count
        else np.nan
    )

    initial_cfg = RC2_WARBURG_INITIAL
    bound_cfg = RC2_WARBURG_BOUNDS

    if (
        np.isfinite(i_pulse_abs)
        and i_pulse_abs > ZERO_CURRENT_LIMIT
        and np.isfinite(v_pulse_tail)
    ):
        r_dyn_guess = abs(v_pulse_tail - ocv_before) / i_pulse_abs - r0_ohm
    else:
        r_dyn_guess = 0.03

    r_dyn_guess = float(np.clip(r_dyn_guess, 0.003, 0.60))
    r1_fraction = float(np.clip(initial_cfg["R1_fraction"], 0.05, 0.85))
    r2_fraction = float(np.clip(initial_cfg["R2_fraction"], 0.05, 0.85))
    rd_fraction = max(1.0 - r1_fraction - r2_fraction, 0.05)

    x0 = np.array([
        r1_fraction * r_dyn_guess,
        r2_fraction * r_dyn_guess,
        rd_fraction * r_dyn_guess,
        float(initial_cfg["tau1"]),
        float(initial_cfg["tau2"]),
        float(initial_cfg["td"]),
        0.0,
    ], dtype=float)

    lower = np.array([
        1e-8,
        1e-8,
        1e-8,
        float(bound_cfg["tau1_min"]),
        float(bound_cfg["tau2_min"]),
        float(bound_cfg["td_min"]),
        -0.030,
    ], dtype=float)

    upper = np.array([
        float(bound_cfg["R1_max"]),
        float(bound_cfg["R2_max"]),
        float(bound_cfg["Rd_max"]),
        float(bound_cfg["tau1_max"]),
        float(bound_cfg["tau2_max"]),
        float(bound_cfg["td_max"]),
        0.030,
    ], dtype=float)

    x0 = np.clip(x0, lower + 1e-12, upper - 1e-12)
    return x0, lower, upper


def _fit_metrics(voltage_true, voltage_pred):
    error_mv = (np.asarray(voltage_pred) - np.asarray(voltage_true)) * 1000.0
    rmse_mv = float(np.sqrt(np.mean(error_mv ** 2)))
    mae_mv = float(np.mean(np.abs(error_mv)))

    ss_res = float(np.sum((voltage_true - voltage_pred) ** 2))
    ss_tot = float(np.sum((voltage_true - np.mean(voltage_true)) ** 2))
    r2_score = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    return rmse_mv, mae_mv, r2_score


def _bound_flags(fit_x, lower, upper, names):
    """标记接近有限边界的参数；无限边界不参与边界命中判断。"""
    fit_x = np.asarray(fit_x, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)

    flags = []
    for value, lo, hi, name in zip(fit_x, lower, upper, names):
        if np.isfinite(lo):
            lower_scale = max(abs(value), abs(lo), 1.0)
            lower_tol = 1e-4 * lower_scale + 1e-10
            if value <= lo + lower_tol:
                flags.append(f"{name}:lower")

        if np.isfinite(hi):
            upper_scale = max(abs(value), abs(hi), 1.0)
            upper_tol = 1e-4 * upper_scale + 1e-10
            if value >= hi - upper_tol:
                flags.append(f"{name}:upper")

    return "；".join(flags) if flags else "无"


def fit_randles_warburg_open(windows, n_terms=5):
    result_rows = []

    for window in windows:
        row = window["row"]
        t_s = window["t_s"]
        current_a = window["current_a"]
        voltage_v = window["voltage_v"]
        weights = window["weights"]
        ocv_before = window["ocv_before"]
        r0_ohm = window["r0_ohm"]

        x0, lower, upper = _warburg_initial_and_bounds(window)

        def residual(param):
            voltage_hat = _simulate_randles_warburg_open(
                param,
                t_s,
                current_a,
                ocv_before,
                r0_ohm,
                n_terms,
            )

            tau1_s = float(param[3])
            tau2_s = float(param[4])
            td_s = float(param[5])

            rc_separation_violation = max(0.0, 3.0 * tau1_s - tau2_s)
            rc_separation_penalty_v = (
                0.002
                * rc_separation_violation
                / max(3.0 * tau1_s, 1e-9)
            )

            penalty_terms = [rc_separation_penalty_v]

            # 默认不再把 td 与 tau2 直接比较。
            # 若仅用于旧结果复现，可临时开启该开关；正式拟合建议保持 False。
            if USE_DIFFUSION_SEPARATION_PENALTY:
                diffusion_separation_violation = max(
                    0.0,
                    1.5 * tau2_s - td_s,
                )
                diffusion_separation_penalty_v = (
                    0.002
                    * diffusion_separation_violation
                    / max(1.5 * tau2_s, 1e-9)
                )
                penalty_terms.append(diffusion_separation_penalty_v)

            return np.concatenate([
                (voltage_hat - voltage_v) * weights,
                np.asarray(penalty_terms, dtype=float),
            ])

        output_row = dict(row)

        try:
            fit = least_squares(
                residual,
                x0=x0,
                bounds=(lower, upper),
                max_nfev=10000,
                x_scale="jac",
                loss="soft_l1",
                f_scale=5e-4,
            )

            (
                r1_ohm,
                r2_ohm,
                rd_ohm,
                tau1_s,
                tau2_s,
                td_s,
                ocv_offset_v,
            ) = fit.x
            voltage_fit = _simulate_randles_warburg_open(
                fit.x,
                t_s,
                current_a,
                ocv_before,
                r0_ohm,
                n_terms,
            )

            rmse_mv, mae_mv, r2_score = _fit_metrics(
                voltage_v, voltage_fit
            )

            if int(n_terms) >= 5:
                voltage_fit_n3_same_param = _simulate_randles_warburg_open(
                    fit.x,
                    t_s,
                    current_a,
                    ocv_before,
                    r0_ohm,
                    WARBURG_CHECK_TERMS,
                )
                truncation_error_mv = (
                    voltage_fit_n3_same_param - voltage_fit
                ) * 1000.0
                n3_n5_trunc_rmse_mv = float(
                    np.sqrt(np.mean(truncation_error_mv ** 2))
                )
                n3_n5_trunc_max_mv = float(
                    np.max(np.abs(truncation_error_mv))
                )
            else:
                n3_n5_trunc_rmse_mv = np.nan
                n3_n5_trunc_max_mv = np.nan

            output_row.update({
                "R1": r1_ohm * 1000.0,
                "R2": r2_ohm * 1000.0,
                "tau1": tau1_s,
                "tau2": tau2_s,
                "C1": tau1_s / r1_ohm if r1_ohm > 0 else np.nan,
                "C2": tau2_s / r2_ohm if r2_ohm > 0 else np.nan,
                "Rd_internal_mOhm": rd_ohm * 1000.0,
                "td_internal_s": td_s,
                "tauW1_internal_s": td_s / (np.pi ** 2),
                "tauW_fastest_retained_s": (
                    td_s / ((int(n_terms) ** 2) * (np.pi ** 2))
                ),
                "tau1_over_tauW1": (
                    tau1_s / (td_s / (np.pi ** 2))
                    if td_s > 0
                    else np.nan
                ),
                "OCV_offset_mV": ocv_offset_v * 1000.0,
                "RMSE_mV": rmse_mv,
                "MAE_mV": mae_mv,
                "R2_score": r2_score,
                "Fit_Success": bool(fit.success),
                "Fit_Status": int(fit.status),
                "Fit_Message": str(fit.message),
                "Fit_NFEV": int(fit.nfev),
                "Fit_Boundary_Flags": _bound_flags(
                    fit.x,
                    lower,
                    upper,
                    [
                        "R1", "R2", "Rd",
                        "tau1", "tau2", "td", "OCV_offset",
                    ],
                ),
                "Warburg_N_Terms": int(n_terms),
                "Warburg_N3_vs_N5_Trunc_RMSE_mV": n3_n5_trunc_rmse_mv,
                "Warburg_N3_vs_N5_Trunc_Max_mV": n3_n5_trunc_max_mv,
                "Truncation_Below_Configured_Noise": (
                    bool(n3_n5_trunc_rmse_mv <= WARBURG_NOISE_FLOOR_MV)
                    if np.isfinite(n3_n5_trunc_rmse_mv)
                    else np.nan
                ),
                "Pulse_Window_s": window["pulse_duration_s"],
                "PAUO_Window_s": window["pauo_duration_s"],
                "Tau2_Exceeds_PAUO_Window": (
                    bool(tau2_s > 2.0 * window["pauo_duration_s"])
                    if window["pauo_duration_s"] > 0
                    else False
                ),
                "Fit_Point_Count": len(t_s),
                "R2_Definition": "R2 = second RC branch resistance",
                "tau2_Definition": "tau2 = second RC branch time constant",
                "C2_Definition": "C2 = tau2 / R2 of second RC branch",
                "Model_Topology": (
                    "R0 + (R1||C1) + (R2||C2) + Warburg Open"
                ),
            })

        except Exception as exc:
            output_row.update({
                "R1": np.nan,
                "R2": np.nan,
                "tau1": np.nan,
                "tau2": np.nan,
                "C1": np.nan,
                "C2": np.nan,
                "Rd_internal_mOhm": np.nan,
                "td_internal_s": np.nan,
                "OCV_offset_mV": np.nan,
                "RMSE_mV": np.nan,
                "MAE_mV": np.nan,
                "R2_score": np.nan,
                "Fit_Success": False,
                "Fit_Status": np.nan,
                "Fit_Message": f"{type(exc).__name__}: {exc}",
                "Fit_NFEV": np.nan,
                "Fit_Boundary_Flags": "拟合异常",
                "Warburg_N_Terms": int(n_terms),
                "Warburg_N3_vs_N5_Trunc_RMSE_mV": np.nan,
                "Warburg_N3_vs_N5_Trunc_Max_mV": np.nan,
                "Truncation_Below_Configured_Noise": np.nan,
                "Pulse_Window_s": window["pulse_duration_s"],
                "PAUO_Window_s": window["pauo_duration_s"],
                "Tau2_Exceeds_PAUO_Window": np.nan,
                "Fit_Point_Count": len(t_s),
                "R2_Definition": "R2 = second RC branch resistance",
                "tau2_Definition": "tau2 = second RC branch time constant",
                "C2_Definition": "C2 = tau2 / R2 of second RC branch",
                "Model_Topology": (
                    "R0 + (R1||C1) + (R2||C2) + Warburg Open"
                ),
            })

        result_rows.append(output_row)

    return pd.DataFrame(result_rows)


In [ ]:
# =============================================================================
# 13. 执行 10% SOC 主拟合：二阶 RC + N=5 Warburg Open
# =============================================================================
# 10% SOC 的筛选已内置在 Cell 12 的 build_stage2_fit_windows() 中。
stage2_windows, stage2_skipped = build_stage2_fit_windows(
    time_diff_sequence,
    r0_result,
)

r1r2_result = fit_randles_warburg_open(
    stage2_windows,
    n_terms=WARBURG_MAIN_TERMS,
)

print(
    f"10% SOC Stage 2 可拟合窗口: {len(stage2_windows)}；"
    f"跳过记录: {len(stage2_skipped)}"
)
print(
    "10% SOC 使用模型："
    "R0 + (R1||C1) + (R2||C2) + finite-length Warburg Open。"
    "R1/R2 和 tau1/tau2 分别对应两个 RC 支路；"
    "Rd_internal_mOhm/td_internal_s 对应 Warburg 参数。"
)

main_display_columns = [
    "SOH",
    "SOC",
    "Time",
    "Zustand",
    "Current",
    "R0_mOhm",
    "R1",
    "R2",
    "Rd_internal_mOhm",
    "tau1",
    "tau2",
    "td_internal_s",
    "tauW1_internal_s",
    "tauW_fastest_retained_s",
    "tau1_over_tauW1",
    "C1",
    "C2",
    "RMSE_mV",
    "R2_score",
    "Fit_Boundary_Flags",
    "Model_Topology",
    "Warburg_N_Terms",
    "Warburg_N3_vs_N5_Trunc_RMSE_mV",
    "Truncation_Below_Configured_Noise",
    "File",
]

main_display_columns = [
    column for column in main_display_columns
    if column in r1r2_result.columns
]

display(
    r1r2_result[main_display_columns]
    .sort_values(
        ["Zustand", "Current", "SOH"],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

warburg_summary = (
    r1r2_result
    .groupby(["SOC", "Model_Topology"], as_index=False)
    .agg(
        Segment_Count=("R2", "size"),
        Fit_Success_Count=("Fit_Success", "sum"),
        Mean_RMSE_mV=("RMSE_mV", "mean"),
        Median_RMSE_mV=("RMSE_mV", "median"),
        Mean_R2_score=("R2_score", "mean"),
        Mean_R2_mOhm=("R2", "mean"),
        Mean_tau2_s=("tau2", "mean"),
        Mean_Rd_mOhm=("Rd_internal_mOhm", "mean"),
        Mean_td_s=("td_internal_s", "mean"),
        Mean_N3_N5_Trunc_RMSE_mV=(
            "Warburg_N3_vs_N5_Trunc_RMSE_mV",
            "mean",
        ),
    )
)

display(warburg_summary)

tau1_upper_count = (
    r1r2_result["Fit_Boundary_Flags"]
    .astype(str)
    .str.contains("tau1:upper", regex=False)
    .sum()
)
tau1_lower_count = (
    r1r2_result["Fit_Boundary_Flags"]
    .astype(str)
    .str.contains("tau1:lower", regex=False)
    .sum()
)
print(
    f"tau1 边界命中：upper={tau1_upper_count}，"
    f"lower={tau1_lower_count}"
)

if not stage2_skipped.empty:
    print("10% SOC Stage 2 跳过记录及原因：")
    display(stage2_skipped)


10% SOC Stage 2 可拟合窗口: 72；跳过记录: 0
10% SOC 使用模型：R0 + (R1||C1) + (R2||C2) + finite-length Warburg Open。R1/R2 和 tau1/tau2 分别对应两个 RC 支路；Rd_internal_mOhm/td_internal_s 对应 Warburg 参数。


,SOH,SOC,Time,Zustand,Current,R0_mOhm,R1,R2,Rd_internal_mOhm,tau1,...,C1,C2,RMSE_mV,R2_score,Fit_Boundary_Flags,Model_Topology,Warburg_N_Terms,Warburg_N3_vs_N5_Trunc_RMSE_mV,Truncation_Below_Configured_Noise,File
0,81.2,10%,2025-07-14 07:38:42.150000+00:00,CHA,1.496636,24.561924,15.910499,4999.233377,20.552578,26.283553,...,1651.962775,14412.070076,0.538486,0.995038,td:lower,R0 + (R1||C1) + (R2||C2) + Warburg Open,5,0.115774,True,METABatt_Sony_Murata_18650VTC6_007_pulse_BM36_...
1,96.2,10%,2024-09-06 06:45:06.580000+00:00,CHA,1.496906,18.759214,15.990373,5000.000000,16.167877,35.009969,...,2189.440462,3893.300565,0.282347,0.995604,R2:upper；td:lower,R0 + (R1||C1) + (R2||C2) + Warburg Open,5,0.068034,True,METABatt_Sony_Murata_18650VTC6_007_pulse_BM4_9...
2,83.9,10%,2025-03-20 03:01:56.710000+00:00,CHA,1.496996,23.451344,14.053451,5000.000000,22.523929,32.376099,...,2303.782855,11744.100327,0.495163,0.995005,R2:upper；td:lower,R0 + (R1||C1) + (R2||C2) + Warburg Open,5,0.127258,True,METABatt_Sony_Murata_18650VTC6_007_pulse_BM26_...
3,81.7,10%,2025-06-27 01:42:05.290000+00:00,CHA,1.498165,25.229957,15.082113,4999.997995,19.702463,27.611460,...,1830.742164,7429.362196,0.519305,0.996802,R2:upper,R0 + (R1||C1) + (R2||C2) + Warburg Open,5,0.136452,True,METABatt_Sony_Murata_18650VTC6_007_pulse_BM34_...
4,78.6,10%,2025-11-10 11:28:15.940000+00:00,CHA,1.498165,26.634383,16.553199,4346.165649,21.139951,24.655479,...,1489.469187,39008.693878,0.590023,0.995147,td:lower,R0 + (R1||C1) + (R2||C2) + Warburg Open,5,0.119951,True,METABatt_Sony_Murata_18650VTC6_007_pulse_BM48_...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,92.2,10%,2024-10-24 06:24:15.220000+00:00,DCH,-2.996265,19.861875,18.698983,2930.248049,24.500455,42.576529,...,2276.943503,1631.767046,0.511556,0.998542,无,R0 + (R1||C1) + (R2||C2) + Warburg Open,5,0.268891,True,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...
68,82.3,10%,2025-06-01 19:14:42.310000+00:00,DCH,-2.995186,24.810461,18.975167,326.874118,17.629011,27.100703,...,1428.219431,3851.487395,0.345422,0.999562,td:lower,R0 + (R1||C1) + (R2||C2) + Warburg Open,5,0.196435,True,METABatt_Sony_Murata_18650VTC6_007_pulse_BM32_...
69,96.2,10%,2024-09-06 07:45:49.870000+00:00,DCH,-2.994826,18.880491,9.512233,4999.926369,113.204854,12.722844,...,1337.524429,1431.379778,0.470010,0.997953,R2:upper,R0 + (R1||C1) + (R2||C2) + Warburg Open,5,0.601824,False,METABatt_Sony_Murata_18650VTC6_007_pulse_BM4_9...
70,85.4,10%,2025-02-13 05:01:57.960000+00:00,DCH,-2.992667,22.543238,18.674106,2067.737951,26.309431,37.749389,...,2021.483011,1934.687733,0.519034,0.998772,无,R0 + (R1||C1) + (R2||C2) + Warburg Open,5,0.280668,True,METABatt_Sony_Murata_18650VTC6_007_pulse_BM22_...


,SOC,Model_Topology,Segment_Count,Fit_Success_Count,Mean_RMSE_mV,Median_RMSE_mV,Mean_R2_score,Mean_R2_mOhm,Mean_tau2_s,Mean_Rd_mOhm,Mean_td_s,Mean_N3_N5_Trunc_RMSE_mV
0,10%,R0 + (R1||C1) + (R2||C2) + Warburg Open,72,72,0.410267,0.4189,0.997885,3139.095875,21495.056777,26.875701,108.4539,0.248226


tau1 边界命中：upper=0，lower=1


In [ ]:
# =============================================================================
# 14. 10% SOC：R0、R1、R2、Rd、tau1、tau2、td 随 SOH 的变化
# =============================================================================
soh_r_table = r1r2_result.copy()

# 提取 CHA / DCH 信息
soh_r_table["Pulse_type"] = (
    soh_r_table["Zustand"]
    .astype(str)
    .str.extract(r"^(CHA|DCH)", expand=False)
)

# 电流取绝对值，避免 DCH 显示成负数
soh_r_table["Current_abs"] = pd.to_numeric(
    soh_r_table["Current"], errors="coerce"
).abs()

# 生成图例标签，例如：CHA 3.0A
soh_r_table["Pulse_label"] = (
    soh_r_table["Pulse_type"].astype(str)
    + " "
    + soh_r_table["Current_abs"].round(1).astype(str)
    + "A"
)

# 排序
soh_r_table = soh_r_table.sort_values(
    ["SOH", "Pulse_type", "Current_abs"]
)

# 每个 pulse 一行，R0/R1/R2/Rd/tau1/tau2/td 在同一张表中
soh_parameter_display = (
    soh_r_table[
        [
            "SOH",
            "SOC",
            "Zustand",
            "Pulse_type",
            "Current",
            "Current_abs",
            "R0_mOhm",
            "R1",
            "R2",
            "Rd_internal_mOhm",
            "tau1",
            "tau2",
            "td_internal_s",
        ]
    ]
    .rename(
        columns={
            "Rd_internal_mOhm": "Rd",
            "td_internal_s": "td",
        }
    )
)

display(soh_parameter_display)

pulse_order = ["CHA 1.5A", "CHA 3.0A", "DCH 3.0A"]

# 与 50% SOC 文件保持一致的固定颜色
color_map = {
    "CHA 1.5A": "#AB63FA",
    "CHA 3.0A": "#FFA15A",
    "DCH 3.0A": "#19D3F3",
}

plot_parameters = [
    ("R0_mOhm", "R0", "R0 (mOhm)"),
    ("R1", "R1", "R1 (mOhm)"),
    ("R2", "R2", "R2 (mOhm)"),
    ("Rd_internal_mOhm", "Rd", "Rd (mOhm)"),
    ("tau1", "tau1", "tau1 (s)"),
    ("tau2", "tau2", "tau2 (s)"),
    ("td_internal_s", "td", "td (s)"),
]

for column_name, display_name, yaxis_title in plot_parameters:

    fig = go.Figure()

    for pulse in pulse_order:
        data = soh_r_table[
            soh_r_table["Pulse_label"] == pulse
        ].sort_values("SOH")

        if data.empty:
            continue

        fig.add_trace(
            go.Scatter(
                x=data["SOH"],
                y=data[column_name],
                mode="lines+markers",
                name=pulse,
                line=dict(color=color_map[pulse], width=2),
                marker=dict(color=color_map[pulse], size=6),
                connectgaps=True,
            )
        )

    fig.update_layout(
        template="plotly_white",
        title=f"SOH vs {display_name} (10% SOC)",
        width=600,
        height=500,
        legend_title_text="pulse / current",
        xaxis_title="SOH (%)",
        yaxis_title=yaxis_title,
    )

    fig.update_xaxes(autorange="reversed")

    fig.show()


,SOH,SOC,Zustand,Pulse_type,Current,Current_abs,R0_mOhm,R1,R2,Rd,tau1,tau2,td
66,77.5,10%,CHA,CHA,1.498795,1.498795,29.035369,15.157606,1851.487157,23.058816,27.972293,78430.862300,30.000648
68,77.5,10%,CHA,CHA,2.997513,2.997513,29.308565,16.041980,31.080364,19.064960,28.322511,436.071398,30.000054
67,77.5,10%,DCH,DCH,-2.997885,2.997885,29.068742,19.335067,220.255691,17.250893,22.557544,938.311459,30.000038
63,77.8,10%,CHA,CHA,1.498345,1.498345,28.742417,15.086985,5000.000000,22.331608,27.687757,106032.869565,30.000000
65,77.8,10%,CHA,CHA,2.994455,2.994455,27.193639,16.758152,29.856603,21.304289,26.478723,407.239093,37.803377
...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,92.2,10%,CHA,CHA,2.999492,2.999492,19.858356,16.941094,2988.863057,22.555316,51.025473,9060.627744,58.871555
70,92.2,10%,DCH,DCH,-2.996265,2.996265,19.861875,18.698983,2930.248049,24.500455,42.576529,4781.482203,104.056322
57,96.2,10%,CHA,CHA,1.496906,1.496906,18.759214,15.990373,5000.000000,16.167877,35.009969,19466.502827,30.000000
59,96.2,10%,CHA,CHA,2.999132,2.999132,18.929873,14.360845,4999.999993,38.247792,90.919608,11816.048951,167.460049
